# MP1 · Prompt Lab — Compare LLM Strategies on a Task

**Rajavarman · Week 5 Graded Mini Project · IITM Pravartak Advanced Certificate Programme in Agentic AI & RAG Engineering**

Compares four prompting strategies (zero-shot, few-shot, structured/role-based, chain-of-thought) on extracting
`company`, `role`, and `years_experience_required` from 10 job posting snippets, scored against a golden set.

Run top to bottom. Requires `OPENAI_API_KEY` set in the environment. See `README.md` in this folder for setup and cost notes.

---

## Setup

In [5]:
import asyncio
import json
import os
import pickle
import re
import time
from pathlib import Path

import pandas as pd
from openai import AsyncOpenAI

# Make sure your OPENAI_API_KEY is set in the environment
assert os.environ.get('OPENAI_API_KEY'), 'Set OPENAI_API_KEY first'

client = AsyncOpenAI()

MODEL = 'gpt-4o-mini'
JUDGE_MODEL = 'gpt-4o'
TEMPERATURE = 0.0

# Cost rates ($ per token) — from W4 cost.py
RATES = {
    'gpt-4o-mini': {'in': 0.15 / 1_000_000, 'out': 0.60 / 1_000_000},
    'gpt-4o':      {'in': 2.50 / 1_000_000, 'out': 10.00 / 1_000_000},
}

print('Setup complete.')

Setup complete.


In [6]:
   resp = await client.chat.completions.create(
       model=MODEL,
       temperature=TEMPERATURE,
       messages=[{"role": "user", "content": "Say hello in one word."}],
   )
   print(resp.choices[0].message.content)

Hello!


## Step 1 — Load the data

In [9]:
DATA_DIR = Path('data')

snippets = [json.loads(line) for line in (DATA_DIR / 'job_snippets.jsonl').read_text().splitlines() if line.strip()]
golden = {row['id']: row for row in (json.loads(line) for line in (DATA_DIR / 'golden_set.jsonl').read_text().splitlines() if line.strip())}

print(f'Loaded {len(snippets)} snippets, {len(golden)} golden entries.')
snippets[0]

Loaded 10 snippets, 10 golden entries.


{'id': 'j01',
 'snippet': 'Acme Corp is hiring a Senior Software Engineer to join our platform team. The ideal candidate has 5+ years of backend development experience and strong skills in Python and distributed systems.'}

## Step 2 — The four prompt strategies

Each strategy is a function that takes the snippet text and returns the `messages` list to send to the model.

All four share the same field instructions (what to extract, and to return `null` rather than guess when a value
isn't stated) so the *only* thing that differs between strategies is the prompting technique itself — the
variable we're isolating, per the brief.

In [10]:
FIELDS_INSTRUCTION = (
    "Extract exactly three fields from the job posting snippet:\n"
    "- company: the company doing the hiring\n"
    "- role: the job title\n"
    "- years_experience_required: the minimum years of experience required, as an integer\n\n"
    "If a field is genuinely not stated in the snippet, use null for that field — "
    "do not infer or guess a value.\n\n"
    "Respond with ONLY a JSON object with keys company, role, years_experience_required. "
    "No other text."
)


def prompt_zero_shot(snippet_text: str) -> list[dict]:
    """Strategy 1 — zero-shot. Just ask, no examples, no persona."""
    return [
        {
            "role": "user",
            "content": f'{FIELDS_INSTRUCTION}\n\nJob posting snippet:\n"""\n{snippet_text}\n"""',
        }
    ]


FEW_SHOT_EXAMPLES = """Example 1:
Snippet: "TechNova is hiring a Backend Engineer. 3+ years of Go experience required."
Output: {"company": "TechNova", "role": "Backend Engineer", "years_experience_required": 3}

Example 2:
Snippet: "Join BrightPath as a Marketing Intern. No prior experience necessary -- we'll train you."
Output: {"company": "BrightPath", "role": "Marketing Intern", "years_experience_required": 0}

Example 3:
Snippet: "Orbital Labs seeks a Staff Scientist. We hire on portfolio strength; no fixed years requirement is listed."
Output: {"company": "Orbital Labs", "role": "Staff Scientist", "years_experience_required": null}"""


def prompt_few_shot(snippet_text: str) -> list[dict]:
    """Strategy 2 — few-shot. Include worked examples covering a clean case, a zero-years case, and a null case."""
    return [
        {
            "role": "user",
            "content": (
                f"{FIELDS_INSTRUCTION}\n\n"
                f"Here are worked examples:\n\n{FEW_SHOT_EXAMPLES}\n\n"
                f'Now do the same for this snippet:\n"""\n{snippet_text}\n"""'
            ),
        }
    ]


STRUCTURED_SYSTEM_PROMPT = """You are an expert technical recruiter with 15 years of experience screening job postings.

Your job is to extract structured data from a job posting snippet. You always follow the schema exactly and never guess a value that isn't supported by the text.

Output schema (JSON):
{
  "company": string,
  "role": string,
  "years_experience_required": integer or null
}

Rules:
- years_experience_required must be the MINIMUM stated years (e.g. "3-5 years" -> 3, "7+ years" -> 7, "around 6 years" -> 6).
- If the snippet does not state a specific years requirement, set years_experience_required to null. Do not infer a number from seniority level, degree requirements, or tone.
- Output ONLY the JSON object. No markdown, no commentary."""


def prompt_structured(snippet_text: str) -> list[dict]:
    """Strategy 3 — structured / role-based. System persona + explicit JSON schema."""
    return [
        {"role": "system", "content": STRUCTURED_SYSTEM_PROMPT},
        {"role": "user", "content": f'Job posting snippet:\n"""\n{snippet_text}\n"""'},
    ]


def prompt_cot(snippet_text: str) -> list[dict]:
    """Strategy 4 — chain-of-thought. Ask the model to reason, then answer with JSON on the final line."""
    return [
        {
            "role": "user",
            "content": (
                f"{FIELDS_INSTRUCTION}\n\n"
                "Think step by step: first identify the company name, then the role title, "
                "then look for any stated minimum years of experience (and note explicitly if none is stated). "
                "Show your brief reasoning, then on the FINAL line output ONLY the JSON object.\n\n"
                f'Job posting snippet:\n"""\n{snippet_text}\n"""'
            ),
        }
    ]


STRATEGIES = {
    "zero_shot": prompt_zero_shot,
    "few_shot": prompt_few_shot,
    "structured": prompt_structured,
    "cot": prompt_cot,
}

## Step 3 — Async batching

Run all 10 snippets × 4 strategies = 40 calls in parallel, capturing strategy, snippet id, raw response,
parsed extraction, cost, and latency for each.

In [11]:
JSON_OBJECT_RE = re.compile(r"\{.*\}", re.DOTALL)


def parse_response(text: str) -> dict | None:
    """Extract a JSON object from the model's response text. Returns None if it doesn't parse.

    Handles ```json ... ``` fences and chain-of-thought prose preceding the JSON block.
    """
    if not text:
        return None
    cleaned = text.strip()
    cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned)
    cleaned = re.sub(r"\s*```$", "", cleaned)
    cleaned = cleaned.strip()

    # For CoT responses there's reasoning before the JSON — grab the last {...} block.
    matches = JSON_OBJECT_RE.findall(cleaned)
    candidate = matches[-1] if matches else cleaned

    try:
        parsed = json.loads(candidate)
    except json.JSONDecodeError:
        return None

    return parsed if isinstance(parsed, dict) else None


async def run_one(strategy_name: str, snippet: dict) -> dict:
    """Run one strategy on one snippet. Returns a dict with all captured fields."""
    messages = STRATEGIES[strategy_name](snippet["snippet"])

    start = time.perf_counter()
    response = await client.chat.completions.create(
        model=MODEL,
        temperature=TEMPERATURE,
        messages=messages,
    )
    latency_s = time.perf_counter() - start

    raw_text = response.choices[0].message.content
    usage = response.usage
    rate = RATES[MODEL]
    cost_usd = usage.prompt_tokens * rate["in"] + usage.completion_tokens * rate["out"]

    return {
        "strategy": strategy_name,
        "snippet_id": snippet["id"],
        "raw_response": raw_text,
        "extracted": parse_response(raw_text),
        "cost_usd": cost_usd,
        "latency_s": latency_s,
    }


async def run_all() -> list[dict]:
    """Run all 10 x 4 = 40 calls in parallel."""
    tasks = [
        run_one(strategy_name, snippet)
        for strategy_name in STRATEGIES
        for snippet in snippets
    ]
    return await asyncio.gather(*tasks)

Results are cached to disk after the first successful run (per the FAQ's caching tip), so re-running this
cell while you iterate on scoring/analysis below doesn't re-spend API budget. Set `USE_CACHE = False` to force
a fresh run (e.g. if you change a prompt).

In [12]:
USE_CACHE = True
CACHE_PATH = Path('mp1_results_cache.pkl')

if USE_CACHE and CACHE_PATH.exists():
    results = pickle.loads(CACHE_PATH.read_bytes())
    print(f'Loaded {len(results)} cached results from {CACHE_PATH}.')
else:
    results = await run_all()
    CACHE_PATH.write_bytes(pickle.dumps(results))
    print(f'Ran {len(results)} calls fresh and cached to {CACHE_PATH}.')

results[0]

Ran 40 calls fresh and cached to mp1_results_cache.pkl.


{'strategy': 'zero_shot',
 'snippet_id': 'j01',
 'raw_response': '{\n  "company": "Acme Corp",\n  "role": "Senior Software Engineer",\n  "years_experience_required": 5\n}',
 'extracted': {'company': 'Acme Corp',
  'role': 'Senior Software Engineer',
  'years_experience_required': 5},
 'cost_usd': 3.87e-05,
 'latency_s': 1.3753545999934431}

## Step 4 — Score against the golden set

Three scores per (strategy × snippet) pair:

1. **accuracy** — how many of the 3 fields match the golden set (0-3), case-insensitive / whitespace-trimmed
   for strings, with `years_experience_required` normalised (`"5+"` -> `5`, `null` stays `null`).
2. **parse_success** — did the response parse cleanly into JSON?
3. **llm_judge_score** — a 1-4 score from `gpt-4o` acting as judge.

In [13]:
def _normalise_str(value) -> str:
    return str(value).strip().lower() if value is not None else ""


def _normalise_years(value):
    """Normalise a years value for comparison: strip a trailing '+', coerce to int, keep None as None."""
    if value is None:
        return None
    if isinstance(value, (int, float)):
        return int(value)
    text = str(value).strip().lower().replace("+", "")
    match = re.search(r"\d+", text)
    return int(match.group()) if match else None


def score_accuracy(extracted: dict | None, gold: dict) -> int:
    """Compare 3 fields (case-insensitive/trimmed strings, normalised years). Returns 0, 1, 2, or 3."""
    if extracted is None:
        return 0

    score = 0
    if _normalise_str(extracted.get("company")) == _normalise_str(gold["company"]):
        score += 1
    if _normalise_str(extracted.get("role")) == _normalise_str(gold["role"]):
        score += 1
    if _normalise_years(extracted.get("years_experience_required")) == _normalise_years(gold["years_experience_required"]):
        score += 1
    return score


JUDGE_PROMPT_TEMPLATE = """You are grading a structured-extraction task. Compare the MODEL OUTPUT to the REFERENCE (gold) answer for the same job posting.

Job posting snippet:
\"\"\"{snippet_text}\"\"\"

Reference (gold) extraction:
{gold_json}

Model output:
{extracted_json}

Score the model output from 1 to 4:
4 - all three fields correct
3 - two of three correct, and no fabricated data
2 - one of three correct, or the model fabricated a field (e.g. invented a number where none was stated)
1 - none correct, or the output did not parse

Respond with ONLY the integer score (1, 2, 3, or 4). No other text."""


async def score_llm_judge(snippet_text: str, extracted: dict | None, gold: dict) -> int:
    """Use gpt-4o as a judge. Returns an integer 1-4."""
    prompt = JUDGE_PROMPT_TEMPLATE.format(
        snippet_text=snippet_text,
        gold_json=json.dumps({k: gold[k] for k in ("company", "role", "years_experience_required")}),
        extracted_json=json.dumps(extracted) if extracted is not None else "null (unparsable)",
    )
    response = await client.chat.completions.create(
        model=JUDGE_MODEL,
        temperature=0.0,
        messages=[{"role": "user", "content": prompt}],
    )
    text = response.choices[0].message.content.strip()
    match = re.search(r"[1-4]", text)
    return int(match.group()) if match else 1

In [14]:
SCORED_CACHE_PATH = Path('mp1_scored_cache.pkl')

if USE_CACHE and SCORED_CACHE_PATH.exists():
    scored = pickle.loads(SCORED_CACHE_PATH.read_bytes())
    print(f'Loaded {len(scored)} cached scored results.')
else:
    snippet_lookup = {s["id"]: s["snippet"] for s in snippets}

    judge_tasks = []
    for row in results:
        gold = golden[row["snippet_id"]]
        row["accuracy"] = score_accuracy(row["extracted"], gold)
        row["parse_success"] = row["extracted"] is not None
        judge_tasks.append(score_llm_judge(snippet_lookup[row["snippet_id"]], row["extracted"], gold))

    judge_scores = await asyncio.gather(*judge_tasks)
    for row, judge_score in zip(results, judge_scores):
        row["llm_judge_score"] = judge_score

    scored = results
    SCORED_CACHE_PATH.write_bytes(pickle.dumps(scored))
    print(f'Scored {len(scored)} results fresh and cached to {SCORED_CACHE_PATH}.')

df = pd.DataFrame(scored)
df.head()

Scored 40 results fresh and cached to mp1_scored_cache.pkl.


,strategy,snippet_id,raw_response,extracted,cost_usd,latency_s,accuracy,parse_success,llm_judge_score
0,zero_shot,j01,"{\n ""company"": ""Acme Corp"",\n ""role"": ""Senio...","{'company': 'Acme Corp', 'role': 'Senior Softw...",0.000039,1.375355,3,True,4
1,zero_shot,j02,"{\n ""company"": ""Northwind Ltd."",\n ""role"": ""...","{'company': 'Northwind Ltd.', 'role': 'Data An...",0.000038,1.427731,2,True,4
2,zero_shot,j03,"{\n ""company"": ""Globex International"",\n ""ro...","{'company': 'Globex International', 'role': 'P...",0.000040,1.637002,3,True,4
3,zero_shot,j04,"{\n ""company"": ""Initech"",\n ""role"": ""Lead De...","{'company': 'Initech', 'role': 'Lead DevOps En...",0.000040,1.366271,3,True,4
4,zero_shot,j05,"{\n ""company"": ""Hooli"",\n ""role"": ""Junior Fr...","{'company': 'Hooli', 'role': 'Junior Frontend ...",0.000039,1.382427,2,True,3


## Step 5 — Build the comparison table

In [15]:
STRATEGY_ORDER = ['zero_shot', 'few_shot', 'structured', 'cot']

summary = df.groupby('strategy').agg(
    **{
        'Accuracy (mean of 3)': ('accuracy', 'mean'),
        'Parse rate': ('parse_success', 'mean'),
        'Judge score (of 4)': ('llm_judge_score', 'mean'),
        'Total cost ($)': ('cost_usd', 'sum'),
        'Latency p50 (s)': ('latency_s', 'median'),
    }
).round(3).reindex(STRATEGY_ORDER)

summary

,Accuracy (mean of 3),Parse rate,Judge score (of 4),Total cost ($),Latency p50 (s)
strategy,,,,,
zero_shot,2.7,1.0,3.9,0.000,1.379
few_shot,3.0,1.0,4.0,0.001,1.377
structured,2.8,1.0,3.9,0.001,1.446
cot,2.7,1.0,3.9,0.001,1.851


## Failure pattern analysis

The rubric specifically calls out the **j10 null-years trap**: j10 states no years requirement, so the gold
`years_experience_required` is `null`. A weak strategy fabricates a number instead of returning `null`. This
cell checks each strategy's behaviour on j10, plus general parse failures and low-scoring rows, to surface the
concrete failure patterns to discuss in the writeup.

In [16]:
j10_rows = df[df['snippet_id'] == 'j10'].copy()
j10_rows['fabricated_years'] = j10_rows['extracted'].apply(
    lambda e: e is not None and e.get('years_experience_required') is not None
)

print("j10 (null-years trap) -- did each strategy fabricate a number?\n")
print(j10_rows[['strategy', 'extracted', 'fabricated_years']].to_string(index=False))

parse_failures = df[~df['parse_success']]
print(f"\nTotal parse failures: {len(parse_failures)}")
if len(parse_failures):
    print(parse_failures[['strategy', 'snippet_id', 'raw_response']].to_string(index=False))

low_accuracy = df[df['accuracy'] <= 1].sort_values(['strategy', 'snippet_id'])
print(f"\nLow-accuracy rows (<=1/3 fields correct): {len(low_accuracy)}")
print(low_accuracy[['strategy', 'snippet_id', 'accuracy', 'extracted']].to_string(index=False))

j10 (null-years trap) -- did each strategy fabricate a number?

  strategy                                                                                               extracted  fabricated_years
 zero_shot {'company': 'Cyberdyne Systems', 'role': 'AI/ML Research Scientist', 'years_experience_required': None}             False
  few_shot {'company': 'Cyberdyne Systems', 'role': 'AI/ML Research Scientist', 'years_experience_required': None}             False
structured {'company': 'Cyberdyne Systems', 'role': 'AI/ML Research Scientist', 'years_experience_required': None}             False
       cot {'company': 'Cyberdyne Systems', 'role': 'AI/ML Research Scientist', 'years_experience_required': None}             False

Total parse failures: 0

Low-accuracy rows (<=1/3 fields correct): 0
Empty DataFrame
Columns: [strategy, snippet_id, accuracy, extracted]
Index: []


### Optional: by-field breakdown

Not required, but a useful stretch — shows whether a strategy's weakness is concentrated in one field
(e.g. strong on `company`/`role` but weak on `years_experience_required`).

In [17]:
def _field_match(row, field):
    extracted = row['extracted']
    gold = golden[row['snippet_id']]
    if extracted is None:
        return False
    if field == 'years_experience_required':
        return _normalise_years(extracted.get(field)) == _normalise_years(gold[field])
    return _normalise_str(extracted.get(field)) == _normalise_str(gold[field])


for field in ['company', 'role', 'years_experience_required']:
    df[f'match_{field}'] = df.apply(lambda r: _field_match(r, field), axis=1)

by_field = df.groupby('strategy')[['match_company', 'match_role', 'match_years_experience_required']].mean().round(3)
by_field = by_field.reindex(STRATEGY_ORDER)
by_field

,match_company,match_role,match_years_experience_required
strategy,,,
zero_shot,0.8,1.0,0.9
few_shot,1.0,1.0,1.0
structured,0.9,1.0,0.9
cot,0.8,1.0,0.9


## Save the comparison table to `mp1_comparison.md`

In [18]:
comparison_md_path = Path('mp1_comparison.md')

md_lines = [
    "# MP1 -- Prompt Strategy Comparison Table",
    "",
    "Model: `gpt-4o-mini` (extraction) - `gpt-4o` (judge) - temperature 0.0",
    "",
    "## Overall comparison",
    "",
    summary.to_markdown(),
    "",
    "## By-field breakdown",
    "",
    by_field.to_markdown(),
    "",
    "## Failure pattern: j10 null-years trap",
    "",
    "j10's snippet states no years requirement; gold `years_experience_required` is `null`. "
    "A strategy that fabricates a number here is hallucinating rather than extracting.",
    "",
    j10_rows[['strategy', 'extracted', 'fabricated_years']].to_markdown(index=False),
]
comparison_md_path.write_text("\n".join(md_lines))
print(f'Wrote {comparison_md_path.resolve()}')

Wrote C:\Users\Raja\Desktop\PYTHON\Week_5_Graded Mini Project_Rajavarman\mp1\mp1_comparison.md


## Step 6 — Write your reflection

Open `mp1_writeup.md` in this folder and answer the four reflection questions **using the numbers produced
above** (the `summary` table, the by-field breakdown, and the j10 / failure-pattern analysis). The template
has guidance for each question but leaves the actual conclusions for you to fill in from your real run.

Then commit:

```bash
git add mp1/
git commit -m "feat(mp1): prompt strategy comparison + writeup"
git push
```